In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_gtfs_calendar = spark.table("bg_traffic.bg_traffic_bronze.gtfs_calendar_dates")
bronze_gtfs_calendar.display()

In [0]:
required_columns = {"service_id", "date", "exception_type"}
missing_columns = required_columns - set(bronze_gtfs_calendar.columns)
if missing_columns:
    raise ValueError("GRESKA: Izvorni GTFS stops je promenio strukturu, postoje nedostajuce kolone!")


In [0]:

bronze_gtfs_calendar.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in bronze_gtfs_calendar.columns]).show()

In [0]:

bronze_gtfs_calendar.printSchema()

### Casting

In [0]:
types_calendar_dates = bronze_gtfs_calendar.select(
    F.col("service_id").cast("string"),
    F.to_date(F.col("date").cast("string"), "yyyyMMdd").alias("date"),
    F.col("exception_type").cast("integer")
)
types_calendar_dates.printSchema()

In [0]:
valid_calendar_dates = types_calendar_dates.dropna(
    subset=["service_id", "date", "exception_type"]
)

In [0]:
types_calendar_dates.display()

### Dedup

In [0]:
dedup_calendar_dates = types_calendar_dates.dropDuplicates(["service_id", "date"])
dedup_count = types_calendar_dates.count() - dedup_calendar_dates.count()
print(f"Broj duplikata: {dedup_count}")

### Valid

In [0]:
valid_calendar_dates = dedup_calendar_dates.filter(
    (F.col("service_id").isNotNull()) &
    F.col("date").isNotNull() 
).withColumn("silver_processed_at", F.current_timestamp())
valid_calendar_dates.display()

In [0]:

if valid_calendar_dates.isEmpty():
    raise Exception("GRESKA: Silver tabela za upisivanje je prazna nakon ciscenja!")

### Write in silver table

In [0]:
valid_calendar_dates.write.format("delta").mode("overwrite").option(
    "overwriteSchema","true"
).saveAsTable("bg_traffic.bg_traffic_silver.gtfs_calendar_date")